In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download
import kagglehub

# Initialize secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")
kaggle_key = user_secrets.get_secret("kaggle_key")

# Authenticate Hugging Face
login(token=hf_token)

# Authenticate Kaggle (Required for uploading)
os.environ['KAGGLE_USERNAME'] = "ash17king0" # Replace with your username
os.environ['KAGGLE_KEY'] = kaggle_key

In [3]:
# Create a temporary directory in /tmp
local_staging_dir = "/tmp/ERNIE-Image-Staging"
os.makedirs(local_staging_dir, exist_ok=True)

print("Downloading 47GB repository to /tmp... this may take a while.")

# The actual download command
snapshot_download(
    repo_id="Comfy-Org/ERNIE-Image", 
    local_dir=local_staging_dir,
    max_workers=8,
    ignore_patterns=[".git*", "README.md"] # Skip unnecessary files to save space
)

print(f"Download complete. Files are at: {local_staging_dir}")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Download complete. Files are at: /tmp/ERNIE-Image-Staging


In [4]:
print("📦 Uploading to Kaggle (This bypasses your 20GB limit)...")
try:
    kagglehub.dataset_upload(
        handle=dataset_handle,
        local_dataset_dir=local_staging_dir
    )
    print(f"✅ Success! Dataset created at kaggle.com/datasets/{dataset_handle}")
except Exception as e:
    print(f"❌ Upload failed: {e}")

📦 Uploading to Kaggle (This bypasses your 20GB limit)...
❌ Upload failed: name 'dataset_handle' is not defined


In [5]:
import os

# 1. Define Paths
# The source is your Dataset (Input)
dataset_path = "/kaggle/input/ernie-image-comfyui"

# The destination is your ComfyUI installation (Working)
base_models_dir = "/kaggle/working/ComfyUI/models"

# 2. Define the subfolder mappings based on your tree
mappings = {
    "diffusion_models": f"{base_models_dir}/diffusion_models",
    "text_encoders": f"{base_models_dir}/text_encoders",
    "vae": f"{base_models_dir}/vae"
}

# 3. Create the directories and links
for folder_name, target_path in mappings.items():
    # Ensure the ComfyUI model folder exists
    os.makedirs(target_path, exist_ok=True)
    
    source_folder = f"{dataset_path}/{folder_name}"
    
    if os.path.exists(source_folder):
        print(f"🔗 Linking {folder_name}...")
        # Link all files from dataset subfolder to ComfyUI subfolder
        !ln -s {source_folder}/* {target_path}/
    else:
        print(f"⚠️ Warning: {folder_name} not found in dataset path!")

print("\n✅ Symlink mapping complete. ComfyUI is ready to load ERNIE-Image.")

⚠️ Warning: diffusion_models not found in dataset path!
⚠️ Warning: text_encoders not found in dataset path!
⚠️ Warning: vae not found in dataset path!

✅ Symlink mapping complete. ComfyUI is ready to load ERNIE-Image.


In [6]:
!ls -l /kaggle/working/ComfyUI/models/diffusion_models/

total 0
